# 02 — Data Cleaning & Preprocessing

This notebook:
1. Fills missing values
2. Combines all text columns into `combined_text`
3. Cleans text (lowercase, remove HTML/URLs/special chars)
4. Applies ADASYN oversampling to fix class imbalance
5. Splits into train/test (80/20)
6. Saves processed data

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

df = pd.read_csv('../datasets/raw/fake_job_postings.csv')
print('Original shape:', df.shape)

Original shape: (17880, 18)


In [2]:
# STEP 1: Fill missing values with empty string
text_cols = ['title', 'location', 'department', 'company_profile',
             'description', 'requirements', 'benefits', 'industry', 'function']
df[text_cols] = df[text_cols].fillna('')
print('Missing values after fill:', df[text_cols].isnull().sum().sum())

Missing values after fill: 0


In [3]:
# STEP 2: Combine all text columns into one 'combined_text' feature
# This is the most important step — gives the model maximum context
df['combined_text'] = (
    df['title'] + ' ' + df['company_profile'] + ' ' +
    df['description'] + ' ' + df['requirements'] + ' ' +
    df['benefits'] + ' ' + df['industry']
)
print('Combined text sample:')
print(df['combined_text'].iloc[0][:200])

Combined text sample:
Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have


In [4]:
# STEP 3: Clean the text
def clean_text(text):
    text = str(text).lower()                      # lowercase everything
    text = re.sub(r'<[^>]+>', ' ', text)          # remove HTML tags
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'[^a-z0-9\s]', ' ', text)     # keep only alphanumeric
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra spaces
    return text

df['cleaned_text'] = df['combined_text'].apply(clean_text)
print('Cleaned text sample:')
print(df['cleaned_text'].iloc[0][:200])

Cleaned text sample:
marketing intern we re food52 and we ve created a groundbreaking and award winning cooking site we support connect and celebrate home cooks and give them everything they need in one place we have a to


In [5]:
# STEP 4: Handle class imbalance using ADASYN
# ADASYN creates synthetic samples of the minority class (fake posts)
from imblearn.over_sampling import ADASYN
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_text']).toarray()
y = df['fraudulent'].values

adasyn = ADASYN(random_state=42)
X_res, y_res = adasyn.fit_resample(X, y)
print('Before ADASYN:', dict(zip(*np.unique(y, return_counts=True))))
print('After ADASYN:', dict(zip(*np.unique(y_res, return_counts=True))))

Before ADASYN: {np.int64(0): np.int64(17014), np.int64(1): np.int64(866)}
After ADASYN: {np.int64(0): np.int64(17014), np.int64(1): np.int64(17090)}


In [6]:
# STEP 5: Train/Test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (27283, 5000), Test: (6821, 5000)


In [7]:
# STEP 6: Save processed data
import os
os.makedirs('../datasets/processed', exist_ok=True)
df[['cleaned_text', 'fraudulent']].to_csv(
    '../datasets/processed/clean_data.csv', index=False)
print('Saved processed dataset!')

Saved processed dataset!
